In [1]:
## pip3 install pyarrow fastparquet matplotlib seaborn
## Go to Kernel --> Restart Kernel after installing the libraries

import json
import pandas as pd
import os
import copy
import duckdb
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import hashlib

ModuleNotFoundError: No module named 'pandas'

In [2]:
## Configure the path for use in the sql-based query
path = '/home/jupyter/2835794-data'

In [3]:
## AD cohort query
## Pulls ICD-10 G30.* (core Alzheimer's disease)
## Plus ICD-9 331.0 for backward compatibility if any legacy data exists
## NOTE: Not filtering by condition_type_concept_id here — pull broad, filter later
##       (Schizoph notebook used 32020 = EHR encounter dx; AD often appears on problem lists too)

query = f"""
SELECT
  per.person_id
  ,per.birth_datetime AS BIRTH_DATE
  ,cond.condition_source_value AS ICD
  ,cond.condition_start_datetime AS START_DATE
  ,cond.condition_type_concept_id AS COND_TYPE
FROM '{path}/person/*.parquet' per
  INNER JOIN '{path}/condition_occurrence/*.parquet' cond
    ON per.person_id = cond.person_id
    AND (
         cond.condition_source_value LIKE 'G30%'      -- ICD-10 Alzheimer's disease (all subtypes)
      OR cond.condition_source_value LIKE 'G30.0%'    -- early onset
      OR cond.condition_source_value LIKE 'G30.1%'    -- late onset
      OR cond.condition_source_value LIKE 'G30.8%'    -- other AD
      OR cond.condition_source_value LIKE 'G30.9%'    -- AD unspecified
      OR cond.condition_source_value = '331.0'        -- ICD-9 Alzheimer's (legacy)
    )
"""
## Note: 'G30%' already covers G30.0/G30.1/G30.8/G30.9 — the subtype LIKEs are
## redundant but kept explicit for readability / easy commenting-out.

In [4]:
## Run query, materialize to pandas
df_ad_raw = duckdb.sql(query).to_df()
df_ad_raw.shape

NameError: name 'duckdb' is not defined

In [5]:
## Quick sanity checks before deciding the cohort definition

# 1. ICD code distribution — which subtypes show up, and at what counts
df_ad_raw['ICD'].value_counts()

NameError: name 'df_ad_raw' is not defined

In [6]:
# 2. condition_type_concept_id distribution
# (32020 = EHR encounter dx, 32840 = EHR problem list, 32817 = EHR claim, etc.)
df_ad_raw['COND_TYPE'].value_counts()

NameError: name 'df_ad_raw' is not defined

In [7]:
# 3. Per-person event counts
events_per_person = df_ad_raw.groupby('person_id').size()
print(f"Unique persons with any AD code: {df_ad_raw['person_id'].nunique()}")
print(f"Median AD codes per person: {events_per_person.median()}")
print(f"Persons with >=2 AD codes: {(events_per_person >= 2).sum()}")

NameError: name 'df_ad_raw' is not defined

In [8]:
# 4. Age at first AD code (sanity: should mostly be >=60, with a tail for early-onset)
df_first = (df_ad_raw
            .sort_values(['person_id', 'START_DATE'])
            .groupby('person_id')
            .first()
            .reset_index())
df_first['AGE_AT_FIRST_AD'] = (
    (pd.to_datetime(df_first['START_DATE']) - pd.to_datetime(df_first['BIRTH_DATE']))
    .dt.days / 365.25
)
df_first['AGE_AT_FIRST_AD'].describe()

NameError: name 'df_ad_raw' is not defined

In [9]:
# 5. Visualize age distribution
plt.figure(figsize=(8, 4))
sns.histplot(df_first['AGE_AT_FIRST_AD'].dropna(), bins=50)
plt.xlabel('Age at first AD code (years)')
plt.ylabel('Count')
plt.title(f'AD cohort: age at first diagnosis (n={len(df_first)})')
plt.axvline(65, color='red', linestyle='--', label='65 (early/late onset cutoff)')
plt.legend()
plt.show()

NameError: name 'plt' is not defined